In [10]:
import pandas as pd
import requests

import folium
import branca.colormap as cm
from folium.plugins import TimestampedGeoJson

from sklearn.cluster import KMeans
from scipy.spatial import ConvexHull

## Загрузка данных

In [11]:
df = pd.read_csv('data/taxi_locations.csv')

df.columns = [c.strip().replace('  ', ' ') for c in df.columns]

for col in ['Trip Start Timestamp', 'Trip End Timestamp']:
    df[col] = pd.to_datetime(df[col], format='%m/%d/%Y %I:%M:%S %p')
    
number_cols = ['Pickup Centroid Latitude', 'Pickup Centroid Longitude',
               'Dropoff Centroid Latitude', 'Dropoff Centroid Longitude', 'Fare']

df[number_cols] = df[number_cols].apply(pd.to_numeric, errors='coerce')
df.head(3)

,Trip ID,Pickup Centroid Latitude,Pickup Centroid Longitude,Pickup Centroid Location,Dropoff Centroid Latitude,Dropoff Centroid Longitude,Dropoff Centroid Location,Trip Start Timestamp,Trip End Timestamp,Taxi ID,Fare
0,304a88bf5d8a7d60c1dbfd88661caa15ee4cb0ea,41.878866,-87.625192,POINT (-87.6251921424 41.8788655841),41.778877,-87.594925,POINT (-87.5949254391 41.7788768603),2014-10-12 20:30:00,2014-10-12 20:45:00,ada14d22a0c2f8c7fe79140611a8e76602acdd8b448282...,19.25
1,b1a8b95ea294663cc3d3cb38a538f464418288c4,41.947792,-87.683835,POINT (-87.6838349425 41.9477915865),41.874005,-87.663518,POINT (-87.6635175498 41.874005383),2014-10-12 20:30:00,2014-10-12 20:45:00,92ab37c26efa3b2659e525a0a2ae31e69c84ffd7bcfef2...,13.05
2,752c8ac4bdf62db4c3d4e810cab915a0ec0ceddc,42.001571,-87.695013,POINT (-87.6950125892 42.001571027),42.001571,-87.695013,POINT (-87.6950125892 42.001571027),2014-09-09 18:00:00,2014-09-09 18:00:00,d06d9962fe86645dd59a7dce41cf27734b23dc62974283...,4.85


## 1. Самые популярные районы

In [12]:
# Кластеризуем точки (подачи или высадки) на k групп и описываем каждую группу.
def cluster_points(coords, k, seed=42):
    # запускаем K-Means: он присваивает каждой точке номер кластера
    model = KMeans(n_clusters=k, random_state=seed, n_init=10)
    labels = model.fit_predict(coords)

    # для каждого кластера собираем: его точки, центр и число поездок
    clusters = []
    for c in range(k):
        pts = coords[labels == c]                 # точки этого кластера
        clusters.append({
            'centroid': pts.mean(axis=0),         # центр кластера (lon, lat)
            'count': len(pts),                    # сколько поездок в кластере
            'points': pts,
        })

    # сортируем от самого крупного кластера к мелкому
    clusters.sort(key=lambda cl: cl['count'], reverse=True)
    return clusters

K = 10
pickup_clusters  = cluster_points(df[['Pickup Centroid Longitude',  'Pickup Centroid Latitude']].values, K)
dropoff_clusters = cluster_points(df[['Dropoff Centroid Longitude', 'Dropoff Centroid Latitude']].values, K)

In [13]:
# Общая карта для всех задач
def base_map(center=(41.88, -87.63), zoom=11):
    return folium.Map(list(center), zoom_start=zoom, tiles='cartodbpositron')

# Граница кластера = выпуклая оболочка его точек
# На одной линии — границы нет
def hull(points):
    try:
        return points[ConvexHull(points).vertices]
    except Exception:
        return None

# Рисуем карту кластеров: границы + центроиды, цвет = число поездок
def cluster_map(clusters, title):
    m = base_map(zoom=10)

    # цветовая шкала: больше поездок -> ярче цвет
    counts = [cl['count'] for cl in clusters]
    colormap = cm.linear.YlOrRd_09.scale(min(counts), max(counts))
    colormap.caption = f'{title}: поездок в кластере'
    colormap.add_to(m)

    for cl in clusters:
        color = colormap(cl['count'])

        # Граница кластера
        border = hull(cl['points'])
        if border is not None:
            # folium ждет [lat, lon], а у нас (lon, lat) — меняем местами
            polygon = [[lat, lon] for lon, lat in border]
            folium.Polygon(polygon, color=color, fill=True,
                           fill_color=color, fill_opacity=0.35).add_to(m)

        # Центроид кластера
        lon, lat = cl['centroid']
        folium.CircleMarker([lat, lon], radius=5, color='black',
                            fill=True, fill_color=color, fill_opacity=1,
                            popup=f'{cl["count"]} поездок').add_to(m)
    return m

map_pickups  = cluster_map(pickup_clusters,  'Подачи')
map_pickups.save('data/map_pickups.html')
map_pickups

In [14]:
map_dropoffs = cluster_map(dropoff_clusters, 'Высадки')
map_dropoffs.save('data/map_dropoffs.html')
map_dropoffs

## 2. Самые популярные маршруты

In [15]:
trip_cols = ['Pickup Centroid Longitude', 'Pickup Centroid Latitude',
             'Dropoff Centroid Longitude', 'Dropoff Centroid Latitude']

# Кластеризуем (координаты старта + координаты финиша) 
# Похожие маршруты попадут в один кластер.
def cluster_trips(df, k, seed=42):
    features = df[trip_cols].values

    model = KMeans(n_clusters=k, random_state=seed, n_init=10)
    labels = model.fit_predict(features)

    routes = []
    for c in range(k):
        trips = features[labels == c]        # поездки этого кластера
        center = trips.mean(axis=0)          # "средняя" поездка кластера (4 числа)
        routes.append({
            'count': len(trips),
            'pickup': center[:2],            # средний старт  (lon, lat)
            'dropoff': center[2:],           # средний финиш (lon, lat)
        })

    routes.sort(key=lambda r: r['count'], reverse=True)
    return routes

K = 25
top_routes = cluster_trips(df, K)[:5]        # 5 самых популярных маршрутов

# Запрашиваем у OSRM реальный путь по улицам между двумя точками
def osrm_route(start, end):                  # start, end = (lon, lat)
    url = (f'https://router.project-osrm.org/route/v1/driving/'
           f'{start[0]},{start[1]};{end[0]},{end[1]}?overview=full&geometries=geojson')
    route = requests.get(url, timeout=10).json()['routes'][0]
    path = [[lat, lon] for lon, lat in route['geometry']['coordinates']]
    return path, route['distance'], route['duration']

# Карта: у каждого топ-маршрута рисуем старт, финиш и путь по улицам
map_routes = base_map(zoom=12)
colors = ['blue', 'purple', 'orange', 'darkred', 'cadetblue']

for i, route in enumerate(top_routes):
    start = tuple(route['pickup'])           # (lon, lat)
    end   = tuple(route['dropoff'])

    try:
        path, dist, dur = osrm_route(start, end)
        folium.PolyLine(path, color=colors[i], weight=4, opacity=0.8,
                        popup=f"Маршрут {i+1}: {route['count']} поездок · "
                              f"{dist/1000:.1f} км · ~{dur/60:.0f} мин").add_to(map_routes)
    except Exception as e:
        straight = [[start[1], start[0]], [end[1], end[0]]]
        folium.PolyLine(straight, color=colors[i], weight=2, dash_array='6').add_to(map_routes)
        print(f'маршрут {i+1}: OSRM недоступен ({type(e).__name__}) — нарисована прямая')

    # центроиды: зеленый старт, красный финиш
    folium.CircleMarker([start[1], start[0]], radius=6, color='green', fill=True, fill_opacity=1,
                        popup=f'Маршрут {i+1}: старт ({route["count"]} поездок)').add_to(map_routes)
    folium.CircleMarker([end[1], end[0]], radius=6, color='red', fill=True, fill_opacity=1,
                        popup=f'Маршрут {i+1}: финиш').add_to(map_routes)

map_routes.save('data/map_routes.html')
map_routes

## 3. Городская инфраструктура

In [16]:
# Ищем точки, куда стекается больше всего поездок
def find_infrastructure(df, top_n=6, freq='h', by='dropoff'):
    # смотрим либо на места высадок, либо на места посадок
    if by == 'dropoff':
        lat_col, lon_col, time_col = 'Dropoff Centroid Latitude', 'Dropoff Centroid Longitude', 'Trip End Timestamp'
    else:
        lat_col, lon_col, time_col = 'Pickup Centroid Latitude', 'Pickup Centroid Longitude', 'Trip Start Timestamp'

    data = df.dropna(subset=[lat_col, lon_col]).copy()
    data['bucket'] = data[time_col].dt.floor(freq)    # округляем время до часа

    # сколько всего поездок в каждой точке -> берем самые загруженные
    totals = data.groupby([lon_col, lat_col]).size().sort_values(ascending=False)
    top_points = totals.head(top_n).index

    rows = []
    for lon, lat in top_points:
        # все поездки этой точки, разбитые по часам
        here = data[(data[lon_col] == lon) & (data[lat_col] == lat)]
        by_hour = here.groupby('bucket').size()

        rows.append({
            'longitude': lon,
            'latitude': lat,
            'num_of_rides': int(by_hour.max()),                       # поездок в час пик
            'Trip End Timestamp': by_hour.idxmax().strftime('%d.%m.%Y %H:%M:%S'),  # сам час пик
        })
    return pd.DataFrame(rows)

rush_hours = find_infrastructure(df, top_n=6, freq='h', by='dropoff')
rush_hours.insert(0, 'name', [f'object_{i+1}' for i in range(len(rush_hours))])
rush_hours.to_csv('data/rush_hours.csv', index=False)   # файл для проверки
rush_hours

,name,longitude,latitude,num_of_rides,Trip End Timestamp
0,object_1,-87.632746,41.880994,891,02.05.2019 09:00:00
1,object_2,-87.620993,41.884987,374,30.05.2019 16:00:00
2,object_3,-87.626215,41.892508,234,20.05.2019 18:00:00
3,object_4,-87.903040,41.979071,263,10.05.2019 13:00:00
4,object_5,-87.642649,41.879255,339,30.04.2019 16:00:00
5,object_6,-87.631864,41.892042,226,31.05.2019 19:00:00


In [17]:
# Достаем координаты точки поездки в виде [lon, lat]
def lonlat(row, kind):                    # kind = 'Pickup' или 'Dropoff'
    return [row[f'{kind} Centroid Longitude'], row[f'{kind} Centroid Latitude']]

# Одна поездка = линия от старта к финишу
def trip_feature(start, end, t0, t1, color, weight=3, opacity=0.8, popup=None):
    props = {'times': [t0.isoformat(), t1.isoformat()],
             'style': {'color': color, 'weight': weight, 'opacity': opacity}}
    if popup:
        props['popup'] = popup
    return {'type': 'Feature',
            'geometry': {'type': 'LineString', 'coordinates': [start, end]},
            'properties': props}

# Точка-маркер, которая появляется в момент времени t
def point_feature(coords, t, color, radius=5, popup=None):
    props = {'times': [t.isoformat()], 'icon': 'circle',
             'iconstyle': {'fillColor': color, 'color': color, 'fillOpacity': 1, 'radius': radius}}
    if popup:
        props['popup'] = popup
    return {'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': coords},
            'properties': props}

# Добавляем на карту анимацию по времени
def add_timeline(m, features, duration, period='PT5M'):
    TimestampedGeoJson({'type': 'FeatureCollection', 'features': features},
                       period=period, duration=duration, transition_time=150,
                       add_last_point=True, loop=False, auto_play=False).add_to(m)
    return m

# Анимация одного дня объекта: кто к нему приезжает и кто уезжает
def animate_location_day(df, lon, lat, date):
    # оставляем поездки только за нужный день
    day = df[df['Trip Start Timestamp'].dt.date == pd.to_datetime(date).date()]

    # поездки, которые закончились в этой точке (люди приехали к объекту)
    inbound = day[(day['Dropoff Centroid Longitude'] == lon) & (day['Dropoff Centroid Latitude'] == lat)]
    # поездки, которые начались в этой точке (люди уехали от объекта)
    outbound = day[(day['Pickup Centroid Longitude'] == lon) & (day['Pickup Centroid Latitude'] == lat)]

    features = []
    # зеленые линии — приезжают к объекту из разных мест
    for _, trip in inbound.iterrows():
        features.append(trip_feature(lonlat(trip, 'Pickup'), [lon, lat],
                                     trip['Trip Start Timestamp'], trip['Trip End Timestamp'], 'green'))
    # оранжевые линии — уезжают от объекта
    for _, trip in outbound.iterrows():
        features.append(trip_feature([lon, lat], lonlat(trip, 'Dropoff'),
                                     trip['Trip Start Timestamp'], trip['Trip End Timestamp'], 'orange'))

    m = base_map((lat, lon))
    # сам объект — красная точка в центре
    folium.CircleMarker([lat, lon], radius=8, color='black',
                        fill=True, fill_color='red', fill_opacity=1, popup='Объект').add_to(m)
    # duration='PT15M' -> поездка гаснет через 15 минут (эффект "нейронов")
    return add_timeline(m, features, duration='PT15M')

In [18]:
# Первый найденный объект и его день часа пик
obj = rush_hours.iloc[0]
date = pd.to_datetime(obj['Trip End Timestamp'], format='%d.%m.%Y %H:%M:%S').date()

map_location = animate_location_day(df, obj['longitude'], obj['latitude'], date)
map_location.save('data/map_neurons.html')
map_location

## 4. Один день из жизни таксиста

In [19]:
# id водителя из задания и единый цвет линий поездок
DRIVER = '2ea4ad2950f3bbdfdcfa7adb48e0dcee49d8a714b7024342f0302eeb9e891dfd55a6f35bb7bc7af06398fb4f55583e1659cb11b432848296bfd2b7d3084e7de1'
TRIP_COLOR = '#1f77b4'

# Анимация дня водителя: поездки не исчезают
def animate_driver_day(df, driver_id, date):
    # поездки этого водителя за нужный день, по порядку завершения
    day = df[(df['Taxi ID'] == driver_id) &
             (df['Trip Start Timestamp'].dt.date == pd.to_datetime(date).date())]
    day = day.sort_values('Trip End Timestamp').reset_index(drop=True)

    if day.empty:                                  # на всякий случай: вдруг поездок нет
        print('Нет поездок у этого водителя в этот день')
        return base_map()

    day['cum'] = day['Fare'].cumsum()              # нарастающий заработок

    features = []
    for i, (_, trip) in enumerate(day.iterrows()):
        start = lonlat(trip, 'Pickup')
        end   = lonlat(trip, 'Dropoff')

        # линия поездки; в подсказке — стоимость и текущий итог
        features.append(trip_feature(start, end, trip['Trip Start Timestamp'], trip['Trip End Timestamp'],
                                     TRIP_COLOR,
                                     popup=f"Поездка #{i+1}: ${trip['Fare']:.2f} · всего ${trip['cum']:.2f}"))
        # зеленый маркер старта
        features.append(point_feature(start, trip['Trip Start Timestamp'], 'green', 5))
        # красный маркер финиша — с текущим заработком (счетчик)
        features.append(point_feature(end, trip['Trip End Timestamp'], 'red', 6,
                                      f"Заработано всего: ${trip['cum']:.2f}"))

    # центр карты — средняя точка поездок дня
    center = (day['Pickup Centroid Latitude'].mean(), day['Pickup Centroid Longitude'].mean())
    # duration=None -> поездки остаются на карте (накапливаются)
    return add_timeline(base_map(center), features, duration=None)

map_driver = animate_driver_day(df, DRIVER, '2019-05-31')
map_driver.save('data/driver_day.html')
map_driver

## 5. Один день в городе

In [20]:
coord_cols = ['Pickup Centroid Longitude', 'Pickup Centroid Latitude',
              'Dropoff Centroid Longitude', 'Dropoff Centroid Latitude']

# Анимация всех поездок города за день (поездки гаснут — эффект "нейронов").
def animate_city_day(df, date, max_trips=4000, seed=42):
    day = df[df['Trip Start Timestamp'].dt.date == pd.to_datetime(date).date()].dropna(subset=coord_cols)

    # Если за день поездок десятки тысяч, берем случайную выборку
    if len(day) > max_trips:
        day = day.sample(max_trips, random_state=seed)

    features = []
    for _, trip in day.iterrows():
        start = lonlat(trip, 'Pickup')
        end   = lonlat(trip, 'Dropoff')
        features.append(trip_feature(start, end, trip['Trip Start Timestamp'], trip['Trip End Timestamp'],
                                     '#e6194b', weight=2, opacity=0.7))

    # duration='PT10M' -> поездки гаснут через 10 минут
    return add_timeline(base_map(), features, duration='PT10M')

map_city = animate_city_day(df, '2019-05-16')
map_city.save('data/city_day.html')
map_city